### Rahven's VENTURI

#### Jolpica Historical Dataset Preparation

This notebook prepares the official Jolpica Formula One historical dataset for downstream analysis.

Pipeline Overview:

1. Configure project paths
2. Locate the official Jolpica archive
3. Extract archive contents
4. Validate extracted files
5. Generate dataset metadata
6. Convert CSV files to Parquet (optional)
7. Save processed datasets
8. Generate preparation logs

This notebook is part of the Rahven's VENTURI Formula One Data Platform.

In [1]:
from pathlib import Path
from zipfile import ZipFile
import shutil
import hashlib
import json
from datetime import datetime

import pandas as pd

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 100)

In [3]:

# Project Root

PROJECT_ROOT = Path.cwd().parent

# Data Directories

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Jolpica Directories

JOLPICA_DIR = RAW_DIR / "jolpica"

ARCHIVE_DIR = JOLPICA_DIR / "archive"
EXTRACTED_DIR = JOLPICA_DIR / "extracted"
METADATA_DIR = JOLPICA_DIR / "metadata"

PROCESSED_JOLPICA_DIR = PROCESSED_DIR / "jolpica"

# Logs

LOG_DIR = PROJECT_ROOT / "logs"

In [4]:
required_directories = [
    ARCHIVE_DIR,
    EXTRACTED_DIR,
    METADATA_DIR,
    PROCESSED_JOLPICA_DIR,
    LOG_DIR,
]

for directory in required_directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Directory structure verified.")

Directory structure verified.


In [5]:
print("Rahven's VENTURI")
print("-" * 50)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Archive      : {ARCHIVE_DIR}")
print(f"Extracted    : {EXTRACTED_DIR}")
print(f"Processed    : {PROCESSED_JOLPICA_DIR}")
print(f"Metadata     : {METADATA_DIR}")
print(f"Logs         : {LOG_DIR}")

Rahven's VENTURI
--------------------------------------------------
Project Root : C:\F1-AI
Archive      : C:\F1-AI\data\raw\jolpica\archive
Extracted    : C:\F1-AI\data\raw\jolpica\extracted
Processed    : C:\F1-AI\data\processed\jolpica
Metadata     : C:\F1-AI\data\raw\jolpica\metadata
Logs         : C:\F1-AI\logs


In [6]:
# Locate Jolpica Archive

zip_files = sorted(ARCHIVE_DIR.glob("*.zip"))

if not zip_files:
    raise FileNotFoundError(
        f"No ZIP archive found in:\n{ARCHIVE_DIR}"
    )

if len(zip_files) > 1:
    print("Multiple archives detected:")
    for file in zip_files:
        print(f"  • {file.name}")

JOLPICA_ZIP = max(
    zip_files,
    key=lambda file: file.stat().st_mtime
)

print(f"Using archive:\n{JOLPICA_ZIP.name}")

Using archive:
jolpica_db_csv.zip


In [7]:
archive_size = JOLPICA_ZIP.stat().st_size / (1024 * 1024)

print("Archive Information")
print("-" * 50)

print(f"Filename : {JOLPICA_ZIP.name}")
print(f"Size     : {archive_size:.2f} MB")
print(f"Modified : {datetime.fromtimestamp(JOLPICA_ZIP.stat().st_mtime)}")

Archive Information
--------------------------------------------------
Filename : jolpica_db_csv.zip
Size     : 13.53 MB
Modified : 2026-07-21 00:23:48.604541


In [8]:
def calculate_sha256(file_path: Path) -> str:
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:

        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


archive_sha256 = calculate_sha256(JOLPICA_ZIP)

print("SHA256")
print("-" * 50)
print(archive_sha256)

SHA256
--------------------------------------------------
672a4b7deb394777506ebe928452df1d5c426354cb51e0ffd0774881fd7e8e6f


In [9]:
with ZipFile(JOLPICA_ZIP, "r") as archive:

    archive_members = archive.namelist()

print(f"Total files inside archive : {len(archive_members)}")

pd.DataFrame({
    "Archive Contents": archive_members
})

Total files inside archive : 18


,Archive Contents
0,formula_one_baseteam.csv
1,formula_one_championshipadjustment.csv
2,formula_one_championshipsystem.csv
3,formula_one_circuit.csv
4,formula_one_driver.csv
5,formula_one_driverchampionship.csv
6,formula_one_lap.csv
7,formula_one_penalty.csv
8,formula_one_pitstop.csv
9,formula_one_pointsystem.csv


In [10]:
# Remove Previous Extraction

if EXTRACTED_DIR.exists():

    shutil.rmtree(EXTRACTED_DIR)

EXTRACTED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Extraction directory cleaned.")

Extraction directory cleaned.


In [11]:
with ZipFile(JOLPICA_ZIP, "r") as archive:

    archive.extractall(EXTRACTED_DIR)

print("Archive extracted successfully.")

Archive extracted successfully.


In [12]:
csv_files = sorted(EXTRACTED_DIR.glob("*.csv"))

print(f"CSV files discovered : {len(csv_files)}")

csv_inventory = pd.DataFrame({

    "Table": [file.stem for file in csv_files],
    "Filename": [file.name for file in csv_files]

})

csv_inventory

CSV files discovered : 18


,Table,Filename
0,formula_one_baseteam,formula_one_baseteam.csv
1,formula_one_championshipadjustment,formula_one_championshipadjustment.csv
2,formula_one_championshipsystem,formula_one_championshipsystem.csv
3,formula_one_circuit,formula_one_circuit.csv
4,formula_one_driver,formula_one_driver.csv
5,formula_one_driverchampionship,formula_one_driverchampionship.csv
6,formula_one_lap,formula_one_lap.csv
7,formula_one_penalty,formula_one_penalty.csv
8,formula_one_pitstop,formula_one_pitstop.csv
9,formula_one_pointsystem,formula_one_pointsystem.csv


In [13]:
expected_tables = {
    "formula_one_baseteam",
    "formula_one_championshipadjustment",
    "formula_one_championshipsystem",
    "formula_one_circuit",
    "formula_one_driver",
    "formula_one_driverchampionship",
    "formula_one_lap",
    "formula_one_penalty",
    "formula_one_pitstop",
    "formula_one_pointsystem",
    "formula_one_round",
    "formula_one_roundentry",
    "formula_one_season",
    "formula_one_session",
    "formula_one_sessionentry",
    "formula_one_team",
    "formula_one_teamchampionship",
    "formula_one_teamdriver",
}

found_tables = {
    file.stem
    for file in csv_files
}

missing_tables = expected_tables - found_tables
extra_tables = found_tables - expected_tables

print(f"Expected : {len(expected_tables)}")
print(f"Found    : {len(found_tables)}")

if missing_tables:
    print("\nMissing Tables")
    print("----------------")

    for table in sorted(missing_tables):
        print(table)

if extra_tables:
    print("\nUnexpected Tables")
    print("-------------------")

    for table in sorted(extra_tables):
        print(table)

if not missing_tables and not extra_tables:
    print("\nDataset validation passed.")

Expected : 18
Found    : 18

Dataset validation passed.


In [14]:
# Load All CSV Tables

tables = {}

for csv_file in sorted(EXTRACTED_DIR.glob("*.csv")):

    table_name = csv_file.stem

    tables[table_name] = pd.read_csv(csv_file)

print(f"Successfully loaded {len(tables)} tables.")

Successfully loaded 18 tables.


In [15]:
dataset_summary = []

for table_name, df in tables.items():

    dataset_summary.append({

        "Table": table_name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2)

    })

summary_df = (
    pd.DataFrame(dataset_summary)
      .sort_values("Table")
      .reset_index(drop=True)
)

summary_df

,Table,Rows,Columns,Memory (MB)
0,formula_one_baseteam,0,3,0.00
1,formula_one_championshipadjustment,3,7,0.00
2,formula_one_championshipsystem,11,10,0.00
3,formula_one_circuit,78,11,0.04
4,formula_one_driver,881,11,0.45
5,formula_one_driverchampionship,36003,14,5.29
6,formula_one_lap,715463,9,178.69
7,formula_one_penalty,0,8,0.00
8,formula_one_pitstop,12672,7,2.60
9,formula_one_pointsystem,24,11,0.01


---------------------------
### **Total Dataset Statistics**

In [16]:
total_rows = sum(df.shape[0] for df in tables.values())
total_columns = sum(df.shape[1] for df in tables.values())
total_memory = sum(
    df.memory_usage(deep=True).sum()
    for df in tables.values()
) / (1024 ** 2)

print("Jolpica Dataset Summary")
print("-" * 50)

print(f"Tables          : {len(tables)}")
print(f"Total Rows      : {total_rows:,}")
print(f"Total Columns   : {total_columns}")
print(f"Memory Usage    : {total_memory:.2f} MB")

Jolpica Dataset Summary
--------------------------------------------------
Tables          : 18
Total Rows      : 867,722
Total Columns   : 165
Memory Usage    : 210.03 MB


-------------------------------
### **Peeking at each table**

In [17]:
for table_name, df in tables.items():

    print("=" * 80)
    print(table_name)
    print("=" * 80)

    display(df.head(3))

formula_one_baseteam


,id,api_id,name


formula_one_championshipadjustment


,id,adjustment,api_id,driver_id,points,season_id,team_id
0,1,101,championshipadjustment_ytJcOgGB,703.0,NaN,48,NaN
1,2,102,championshipadjustment_yy7ogVVY,NaN,NaN,58,117.0
2,3,1,championshipadjustment_zXL6omRH,NaN,15.0,71,208.0


formula_one_championshipsystem


...

,id,api_id,base_team_id,country_code,name,nationality,primary_color,reference,wikipedia
0,1,team_1XDKqgNu,NaN,CHE,Alfa Romeo,Swiss,NaN,alfa,https://en.wikipedia.org/wiki/Alfa_Romeo_in_Formula_One
1,2,team_e9i5UU0k,NaN,ITA,Maserati,Italian,NaN,maserati,https://en.wikipedia.org/wiki/Maserati
2,3,team_R6ZNEH9U,NaN,GBR,Alta,British,NaN,alta,https://en.wikipedia.org/wiki/Alta_Car_and_Engineering_Company


formula_one_teamchampionship


,id,adjustment_type,highest_finish,is_eligible,points,position,round_id,round_number,season_id,session_id,session_number,team_id,win_count,year
0,197289,0,NaN,t,0.0,1.0,NaN,1,NaN,5134,4,191,0,2026
1,197290,0,NaN,t,0.0,2.0,NaN,1,NaN,5134,4,141,0,2026
2,197291,0,NaN,t,0.0,3.0,NaN,1,NaN,5134,4,64,0,2026


formula_one_teamdriver


,id,api_id,driver_id,role,season_id,team_id
0,1,teamdriver_LSox06Od,1,NaN,1,1
1,2,teamdriver_bLowFOCJ,2,NaN,1,1
2,3,teamdriver_d8PF1G74,3,NaN,1,1


----------------------------
### **Validate every table**

In [18]:
validation_summary = []

for table_name, df in tables.items():

    duplicate_rows = int(df.duplicated().sum())

    null_cells = int(df.isna().sum().sum())

    has_id = "id" in df.columns

    duplicate_ids = (
        int(df["id"].duplicated().sum())
        if has_id
        else None
    )

    null_ids = (
        int(df["id"].isna().sum())
        if has_id
        else None
    )

    validation_summary.append({
        "Table": table_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Null Cells": null_cells,
        "Duplicate Rows": duplicate_rows,
        "Duplicate IDs": duplicate_ids,
        "Null IDs": null_ids,
        "Memory (MB)": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            2
        ),
    })

validation_df = (
    pd.DataFrame(validation_summary)
      .sort_values("Table")
      .reset_index(drop=True)
)

validation_df

,Table,Rows,Columns,Null Cells,Duplicate Rows,Duplicate IDs,Null IDs,Memory (MB)
0,formula_one_baseteam,0,3,0,0,0,0,0.00
1,formula_one_championshipadjustment,3,7,5,0,0,0,0.00
2,formula_one_championshipsystem,11,10,0,0,0,0,0.00
3,formula_one_circuit,78,11,0,0,0,0,0.04
4,formula_one_driver,881,11,1640,0,0,0,0.46
5,formula_one_driverchampionship,36003,14,53769,0,0,0,5.29
6,formula_one_lap,715463,9,809682,0,0,0,178.69
7,formula_one_penalty,0,8,0,0,0,0,0.00
8,formula_one_pitstop,12672,7,0,0,0,0,2.60
9,formula_one_pointsystem,24,11,0,0,0,0,0.01


**Convert csv --> Parquet**

In [19]:
for table_name, df in tables.items():

    output_path = PROCESSED_JOLPICA_DIR / f"{table_name}.parquet"

    df.to_parquet(
        output_path,
        index=False
    )

print(f"Converted {len(tables)} tables to Parquet.")

Converted 18 tables to Parquet.


In [20]:
metadata = {
    "created_at": datetime.now().isoformat(),
    "archive_name": JOLPICA_ZIP.name,
    "archive_sha256": archive_sha256,
    "table_count": len(tables),
    "total_rows": total_rows,
    "total_columns": total_columns,
    "total_memory_mb": round(total_memory, 2),
}

metadata_path = METADATA_DIR / "dataset_metadata.json"

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)

print(f"Metadata saved:\n{metadata_path}")

Metadata saved:
C:\F1-AI\data\raw\jolpica\metadata\dataset_metadata.json


### **Final Pipeline Summary**

In [22]:
print("=" * 60)
print("Rahven's VENTURI - Jolpica Preparation Complete")
print("=" * 60)

print(f"Archive           : {JOLPICA_ZIP.name}")
print(f"Tables Loaded     : {len(tables)}")
print(f"Total Rows        : {total_rows:,}")
print(f"Processed Tables  : {len(tables)} Parquet files")
print(f"Metadata Folder   : {METADATA_DIR}")
print(f"Processed Folder  : {PROCESSED_JOLPICA_DIR}")

print("\nNext Notebook:")
print("04_jolpica_eda.ipynb")

Rahven's VENTURI - Jolpica Preparation Complete
Archive           : jolpica_db_csv.zip
Tables Loaded     : 18
Total Rows        : 867,722
Processed Tables  : 18 Parquet files
Metadata Folder   : C:\F1-AI\data\raw\jolpica\metadata
Processed Folder  : C:\F1-AI\data\processed\jolpica

Next Notebook:
04_jolpica_eda.ipynb


In [28]:
# -----------------------------
# Generate Schema Report
# -----------------------------

schema_rows = []

for table_name, df in tables.items():

    for column in df.columns:

        schema_rows.append({
            "Table": table_name,
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Non-Null Count": int(df[column].count()),
            "Null Count": int(df[column].isna().sum()),
            "Unique Values": int(df[column].nunique(dropna=True))
        })

schema_df = (
    pd.DataFrame(schema_rows)
      .sort_values(["Table", "Column"])
      .reset_index(drop=True)
)

schema_path = METADATA_DIR / "schema_report.csv"

schema_df.to_csv(schema_path, index=False)

print(f"Schema report saved to:\n{schema_path}")

print(schema_df.head(20))
print(schema_df.shape)

Schema report saved to:
C:\F1-AI\data\raw\jolpica\metadata\schema_report.csv
                                 Table                   Column Data Type  Non-Null Count  Null Count  Unique Values
0                 formula_one_baseteam                   api_id    object               0           0              0
1                 formula_one_baseteam                       id    object               0           0              0
2                 formula_one_baseteam                     name    object               0           0              0
3   formula_one_championshipadjustment               adjustment     int64               3           0              3
4   formula_one_championshipadjustment                   api_id    object               3           0              3
5   formula_one_championshipadjustment                driver_id   float64               1           2              1
6   formula_one_championshipadjustment                       id     int64               3           0   